# 🔄 Módulo 09 — Orchestration: Sequential

> **Objetivo:** Construir un pipeline de agentes donde cada uno recibe el output del anterior.

📚 **Doc oficial:** [Sequential Orchestration (Python)](https://learn.microsoft.com/en-us/agent-framework/workflows/orchestrations/sequential?pivots=programming-language-python)

## ¿Qué es Sequential Orchestration?

Los agentes se organizan en **pipeline** — cada uno procesa la tarea por turno y pasa su output al siguiente.

```
Usuario  ──→  [Analista]  ──→  [Writer]  ──→  [Editor]  ──→  Anuncio final
```

## Escenario: **Crear un anuncio publicitario para un producto**

| Agente | Rol |
|--------|-----|
| 🔍 **Analista** | Extrae los elementos clave del producto (material, tamaño, uso, público) |
| ✏️ **Writer** | Escribe un texto publicitario basado en esos elementos |
| ✨ **Editor** | Pule y mejora el texto publicitario final |

## API clave

```python
from agent_framework.orchestrations import SequentialBuilder

workflow = SequentialBuilder(participants=[analista, writer, editor]).build()
result = await workflow.run("Botella de 500ml de metal...")
```


## ⚙️ Setup

Carga el ChatClient compartido.

In [6]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados.")


✅ Helpers cargados.


## 1️⃣ Pipeline básico: Analista → Writer

El **Analista** extrae los elementos clave del producto y el **Writer** los convierte en un texto publicitario.

> 💡 En este ejemplo **no** usamos `intermediate_output_from`, así que solo vemos el resultado final
> con `get_outputs()`. Internamente el Analista sí trabajó, pero su output solo fue visible para el Writer.


In [9]:
from agent_framework import AgentResponse, Message
from agent_framework.orchestrations import SequentialBuilder

client = create_chat_client()

analista = Agent(
    client=client,
    name="analista",
    instructions=(
        "Eres un analista de producto. Cuando recibes la descripción de un producto, "
        "extrae los elementos clave en este formato:\n"
        "- Producto: ...\n"
        "- Material: ...\n"
        "- Tamaño/Capacidad: ...\n"
        "- Público objetivo: ...\n"
        "- Beneficios principales: ...\n"
        "- Diferenciador: ...\n"
        "Sé conciso. Solo lista los elementos, sin texto extra."
    ),
)

writer = Agent(
    client=client,
    name="writer",
    instructions=(
        "Eres un redactor publicitario. Recibes una lista de elementos clave de un producto. "
        "Escribe un texto publicitario corto (3-4 oraciones) que destaque los beneficios principales. "
        "Tono: moderno, directo, aspiracional."
    ),
)

# --- Pipeline sin intermediate_output_from ---
# Solo el último agente (writer) aparece en get_outputs().
# El analista trabaja internamente y su resultado pasa al writer.
workflow = SequentialBuilder(participants=[analista, writer]).build()

# Ejecutamos sin streaming — esperamos el resultado completo
result = await workflow.run("Botella de 500ml de metal, doble pared, mantiene bebidas frías 24h y calientes 12h.")

# get_outputs() retorna una lista con los outputs del workflow.
# Cada output puede ser:
#   - Una lista de Message (la conversación completa entre agentes)
#   - Un solo AgentResponse (la respuesta del último agente)
# Por eso normalizamos: si ya es lista la usamos directo, si no la envolvemos en una.
outputs = result.get_outputs()

if outputs:
    conversation = outputs[0] if isinstance(outputs[0], list) else [outputs[0]]

    for msg in conversation:
        # Cada mensaje tiene .author_name (el nombre del agente que lo generó)
        # y .text (el contenido de la respuesta)
        name = getattr(msg, "author_name", None) or "assistant"
        emoji = "🔍" if name == "analista" else "✏️"
        print(f"{emoji} [{name}]:\n{msg.text}\n")


✏️ [assistant]:
Mantén tu bebida en su punto perfecto, estés donde estés. Nuestra botella de 500 ml con doble pared conserva lo frío por 24 horas y lo caliente por 12. Ligera, resistente y con estilo, es el accesorio imprescindible para tu ritmo imparable. ¡Hidratación inteligente, sin límites!



## 2️⃣ Pipeline completo con streaming: Analista → Writer → Editor

Ahora sumamos al **Editor** que mejora el texto publicitario.

### `intermediate_output_from` — ver lo que hace cada agente

Por defecto, solo el **último agente** del pipeline emite su respuesta como evento `"output"`.
Los agentes anteriores trabajan "en silencio" — su resultado pasa al siguiente, pero tú no lo ves.

Con `intermediate_output_from` puedes "espiar" lo que producen los agentes intermedios:

| Valor | Qué hace |
|-------|----------|
| `None` (default) | Solo ves el output del último agente |
| `"all_other"` | Ves el output de **todos excepto** el último (que ya llega como `"output"`) |
| `[agent_a, agent_b]` | Ves solo los agentes específicos que listes |

Los eventos intermedios llegan con `event.type == "intermediate"`, mientras que el final llega como `"output"`.

### `stream=True` — ver el texto en tiempo real

Con streaming, recibes `AgentResponseUpdate` con fragmentos de texto a medida que cada agente genera su respuesta.


In [10]:
# AgentResponseUpdate es el tipo de dato que recibimos en cada fragmento de streaming.
# Contiene `.text` con el texto parcial que el agente va generando.
from agent_framework import AgentResponseUpdate

editor = Agent(
    client=client,
    name="editor",
    instructions=(
        "Eres un editor publicitario senior. Recibes un texto publicitario y lo mejoras:\n"
        "- Hazlo más impactante y memorable\n"
        "- Asegúrate de que sea claro y fluido\n"
        "- Agrega un call-to-action al final\n"
        "Devuelve SOLO el texto publicitario mejorado, nada más."
    ),
)

# --- Construimos el workflow ---
# intermediate_output_from="all_other":
#   → El "analista" y "writer" emiten eventos "intermediate" (podemos ver su trabajo)
#   → El "editor" (último) emite el evento "output" final
workflow = SequentialBuilder(
    participants=[analista, writer, editor],
    intermediate_output_from="all_other",
).build()

producto = "Mochila urbana con panel solar integrado, carga tu celular mientras caminas, resistente al agua, 25 litros."

print(f"📦 PRODUCTO: {producto}")
print("=" * 60)

# --- Streaming: recibimos fragmentos de texto en tiempo real ---
#
# ¿Cómo funciona el streaming?
# El modelo NO devuelve todo el texto de golpe. Lo devuelve en pedacitos (tokens).
# Por ejemplo, si el analista responde "- Producto: Mochila solar", recibirás algo como:
#   evento 1: "- Producto"
#   evento 2: ": Mochila"
#   evento 3: " solar"
#
# Cada evento trae:
#   - event.type → "intermediate" (analista, writer) o "output" (editor, el último)
#   - event.data → un AgentResponseUpdate con .text (el pedacito de texto)
#   - event.data.author_name → el nombre del agente que está hablando

last_author = None  # Guardamos quién habló por última vez para detectar cambios de agente

async for event in workflow.run(producto, stream=True):
    if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = getattr(update, "author_name", None) or event.type

        # --- ¿Cambió de agente? ---
        # Ejemplo: estábamos recibiendo tokens del "analista" y ahora llega uno del "writer"
        # Eso significa que el analista terminó y el writer empezó a trabajar.
        if author != last_author:
            # Salto de línea para separar visualmente del agente anterior
            if last_author is not None:
                print("\n")

            # Imprimimos el encabezado del nuevo agente: "✏️ [writer]"
            emojis = {"analista": "🔍", "writer": "✏️", "editor": "✨"}
            print(f"{emojis.get(author, '📝')} [{author}]")

            # Imprimimos el primer pedacito de texto de este agente
            # end="" → NO saltar línea, porque el siguiente token se concatena al lado
            # flush=True → mostrar inmediatamente, no esperar a que se llene el buffer
            print(update.text, end="", flush=True)
            last_author = author
        else:
            # --- Mismo agente, siguiente token ---
            # El agente sigue hablando, simplemente concatenamos el pedacito al lado
            # Ejemplo: ya imprimimos "- Producto", ahora llega ": Mochila" → queda "- Producto: Mochila"
            print(update.text, end="", flush=True)

print("\n\n✅ Analista extrajo elementos → Writer creó el anuncio → Editor lo pulió.")


📦 PRODUCTO: Mochila urbana con panel solar integrado, carga tu celular mientras caminas, resistente al agua, 25 litros.
🔍 [analista]
- Producto: Mochila urbana con panel solar integrado  
- Material: Resistente al agua (no especificado)  
- Tamaño/Capacidad: 25 litros  
- Público objetivo: Personas urbanas, tech-savvy, aventureros  
- Beneficios principales: Carga del celular mientras caminas, resistencia al agua  
- Diferenciador: Panel solar integrado

✏️ [writer]
Libera tu energía con la mochila urbana que lo tiene todo. Su panel solar integrado carga tu celular mientras conquistas la ciudad, y su diseño resistente al agua protege lo que más importa. Con 25 litros de capacidad, estilo y funcionalidad se encuentran en cada paso. ¿Listo para llevar tu vida al siguiente nivel?

✨ [editor]
Carga tus días con estilo y tecnología. Nuestra mochila urbana con panel solar integrado te permite cargar tu celular mientras te mueves. Resistente al agua, con 25 litros de capacidad y un diseño pen

## 3️⃣ `chain_only_agent_responses=True` — cada agente solo ve al anterior

Por defecto cada agente ve **toda la conversación** (producto original + respuestas previas).
Con `chain_only_agent_responses=True`, cada agente **solo recibe la respuesta del anterior**.

Aquí lo usamos para una **cadena de traducción**: español → inglés → francés.


In [11]:
# --- Traductores ---
traductor_ingles = Agent(
    client=client,
    name="traductor_ingles",
    instructions="Traduce el texto al inglés. Responde SOLO con la traducción.",
)

traductor_frances = Agent(
    client=client,
    name="traductor_frances",
    instructions="Traduce el texto al francés. Responde SOLO con la traducción.",
)

# chain_only_agent_responses=True:
#   → Cada agente SOLO ve la respuesta del anterior, NO el prompt original.
#   → Ideal para traducción: el francés solo ve el inglés, no el español.
#
# intermediate_output_from="all_other":
#   → Nos permite ver la traducción al inglés (intermedia) además de la francesa (final).
workflow = SequentialBuilder(
    participants=[traductor_ingles, traductor_frances],
    chain_only_agent_responses=True,
    intermediate_output_from="all_other",
).build()

last_author = None
async for event in workflow.run("El futuro pertenece a quienes se preparan hoy.", stream=True):
    if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = getattr(update, "author_name", None) or event.type
        if author != last_author:
            if last_author is not None:
                print("\n")
            emojis = {"traductor_ingles": "🇬🇧", "traductor_frances": "🇫🇷"}
            print(f"{emojis.get(author, '📝')} [{author}]")
            print(update.text, end="", flush=True)
            last_author = author
        else:
            print(update.text, end="", flush=True)

print("\n\n✅ Español → Inglés → Francés (cada agente solo vio la respuesta del anterior)")


🇬🇧 [traductor_ingles]
The future belongs to those who prepare today.

🇫🇷 [traductor_frances]
L'avenir appartient à ceux qui se préparent aujourd'hui.

✅ Español → Inglés → Francés (cada agente solo vio la respuesta del anterior)


## 4️⃣ Human-in-the-Loop — pausar el pipeline para revisión humana

En algunos flujos no quieres que el pipeline corra sin supervisión.
Con `.with_request_info(agents=[...])` puedes **pausar** el workflow después de que un agente específico responda,
permitiendo que un humano revise, apruebe o dé feedback antes de continuar.

### ¿Cómo funciona?

1. El workflow corre normalmente hasta llegar al agente marcado
2. Ese agente genera su respuesta
3. El workflow **se pausa** y emite un evento `"request_info"` con un `request_id`
4. Tú decides si aprobar (`AgentRequestInfoResponse.approve()`) o dar feedback
5. Llamas a `workflow.run(stream=True, responses={request_id: respuesta})` para **reanudar**

```
Usuario → [Analista] → [Writer] → ⏸️ PAUSA (humano revisa) → [Editor] → Anuncio final
```


In [ ]:
from agent_framework.orchestrations import AgentRequestInfoResponse

# --- Reutilizamos los agentes: analista → writer → editor ---
# Pero esta vez pausamos después del "writer" para que un humano revise
# el texto ANTES de que el editor lo pula.

workflow = (
    SequentialBuilder(
        participants=[analista, writer, editor],
        intermediate_output_from="all_other",
    )
    # with_request_info(agents=["writer"]):
    #   → Después de que el writer responda, el workflow se PAUSA
    #   → Emite un evento "request_info" y espera nuestra respuesta
    #   → Solo entonces continúa con el editor
    .with_request_info(agents=["writer"])
    .build()
)

producto = "Audífonos inalámbricos con cancelación de ruido activa, 40h de batería, carga rápida de 10 min para 3h de uso."

print(f"📦 PRODUCTO: {producto}")
print("=" * 60)


# --- Función para procesar eventos del stream ---
# Recorre los eventos y:
#   1. Imprime los outputs intermedios y finales (como en el ejercicio 2)
#   2. Si encuentra un evento "request_info", guarda el request_id y la respuesta de aprobación
async def process_event_stream(stream):
    responses = {}
    last_author = None

    async for event in stream:
        # --- Outputs de los agentes (streaming) ---
        if event.type in ("output", "intermediate") and isinstance(event.data, AgentResponseUpdate):
            update = event.data
            author = getattr(update, "author_name", None) or event.type
            if author != last_author:
                if last_author is not None:
                    print("\n")
                emojis = {"analista": "🔍", "writer": "✏️", "editor": "✨"}
                print(f"{emojis.get(author, '📝')} [{author}]")
                print(update.text, end="", flush=True)
                last_author = author
            else:
                print(update.text, end="", flush=True)

        # --- Evento de pausa: el workflow pide aprobación humana ---
        elif event.type == "request_info":
            print("\n")
            print("⏸️  === PAUSA: El writer terminó. ¿Aprobamos para continuar con el editor? ===")
            print("    (En una app real aquí mostrarías UI para que el humano revise)")
            print("    ✅ Aprobando automáticamente para continuar...\n")

            # Guardamos la respuesta: approve() indica que el humano aprueba
            # El request_id vincula esta respuesta con el evento que la pidió
            responses[event.request_id] = AgentRequestInfoResponse.approve()

    return responses if responses else None


# --- Ejecutamos el workflow ---
# Primera ejecución: corre analista → writer → PAUSA
stream = workflow.run(producto, stream=True)
pending_responses = await process_event_stream(stream)

# Si hay respuestas pendientes, reanudamos el workflow
# Segunda ejecución: recibe la aprobación y corre el editor
while pending_responses is not None:
    stream = workflow.run(stream=True, responses=pending_responses)
    pending_responses = await process_event_stream(stream)

print("\n✅ Pipeline con human-in-the-loop completado.")
print("   El workflow se pausó después del writer y continuó tras la aprobación.")


## 🎯 Resumen

| Capacidad | API |
|-----------|-----|
| Pipeline básico | `SequentialBuilder(participants=[a, b]).build()` |
| Streaming | `async for event in workflow.run(prompt, stream=True):` |
| Outputs intermedios | `intermediate_output_from="all_other"` |
| Solo ver respuesta anterior | `chain_only_agent_responses=True` |
| Pausar para revisión humana | `.with_request_info(agents=["nombre"])` |
| Aprobar y continuar | `AgentRequestInfoResponse.approve()` |

**Siguiente módulo →** [Módulo 10 — Orchestration Concurrent](./10_orchestration_concurrent.ipynb)
